# MISPR Workshop — DFT & MD Workflows
### Stony Brook University · MolMD Group

Three tutorials — **ESP** (partial charges), **Binding energy**, and **MD** — run entirely in Google Colab. You need only a Google account: no installs, no licenses, no payment.

**Workshop flow:** run the **Setup** cell once (~2 min). Then go through each section one cell at a time — read the explanation, run the cell, look at the output — just like the original Code Ocean capsule.

*The tutorials replay pre-computed Gaussian/LAMMPS outputs (`run_fake_gaussian` / `run_fake_lammps`), so each step finishes in seconds on a free CPU runtime.*

## Setup — run this one cell first (~2 min)

In [ ]:
#@title ▶ Set up the workshop environment { display-mode: "form" }
# ============================================================================
GITHUB_REPO = "https://github.com/kuldeepsinhraj/mispr-workshop-colab"
ENV_URL  = "https://github.com/kuldeepsinhraj/mispr-workshop-colab/releases/download/v1.0/mispr-env.tar.gz"
DATA_URL = "https://github.com/kuldeepsinhraj/mispr-workshop-colab/releases/download/v1.0/data.zip"
# ============================================================================
import os, time, shutil, subprocess
CONDA, CAPSULE, SRVDIR = "/opt/conda", "/root/capsule", "/tmp/mispr_srv"
def _sh(c): subprocess.run(c, shell=True, check=True)
t0 = time.time(); os.makedirs(CAPSULE, exist_ok=True)

print("[1/5] Downloading the Python 3.8 environment (~1.8 GB)...", flush=True)
if not os.path.exists(CONDA + "/bin/python"):
    _sh("wget -q -O /tmp/env.tar.gz '" + ENV_URL + "'")
    _sh("tar -xzf /tmp/env.tar.gz -C /")
    subprocess.run(CONDA + "/bin/conda-unpack", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    _sh("rm -f /tmp/env.tar.gz")

print("[2/5] Fetching code and reference data...", flush=True)
if not os.path.exists(CAPSULE + "/code"):
    _sh("git clone --depth 1 -q '" + GITHUB_REPO + "' /tmp/repo")
    _sh("cp -r /tmp/repo/code " + CAPSULE + "/code")
if not os.path.isdir(CAPSULE + "/data/be"):
    shutil.rmtree("/tmp/dz", ignore_errors=True)
    fname = DATA_URL.split("/")[-1]
    _sh("wget -q -O /tmp/" + fname + " '" + DATA_URL + "'")
    shutil.unpack_archive("/tmp/" + fname, "/tmp/dz")
    root = None
    for r, d, f in os.walk("/tmp/dz"):
        if all(os.path.isdir(os.path.join(r, x)) for x in ["be", "esp", "md"]):
            root = r; break
    if not root: raise SystemExit("data.zip does not contain be/ esp/ md/.")
    os.makedirs(CAPSULE + "/data", exist_ok=True)
    for x in ["be", "esp", "md"]:
        if not os.path.exists(CAPSULE + "/data/" + x):
            shutil.copytree(os.path.join(root, x), CAPSULE + "/data/" + x)
os.makedirs(CAPSULE + "/results", exist_ok=True)
need = [CAPSULE+"/data/esp/water/ESP", CAPSULE+"/data/be/co2/Optimization",
        CAPSULE+"/data/md/runs", CAPSULE+"/data/md/result/npt/log.lammps"]
miss = [p for p in need if not os.path.exists(p)]
if miss: raise SystemExit("Missing reference data:\n  " + "\n  ".join(miss))

print("[3/5] Starting MongoDB...", flush=True)
os.makedirs(CAPSULE + "/db", exist_ok=True)
menv = os.environ.copy(); menv["LD_LIBRARY_PATH"] = CONDA + "/lib:" + menv.get("LD_LIBRARY_PATH","")
subprocess.run([CONDA+"/bin/mongod","--dbpath",CAPSULE+"/db","--bind_ip","127.0.0.1",
                "--port","27017","--logpath",CAPSULE+"/db/mongod.log","--fork"],
               env=menv, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):
    if subprocess.run("ss -ltn 2>/dev/null | grep -q 27017", shell=True).returncode == 0: break
    time.sleep(1)
else: raise SystemExit("MongoDB failed — see " + CAPSULE + "/db/mongod.log")

print("[4/5] Starting the MISPR (Python 3.8) helper...", flush=True)
# A persistent Python-3.8 process. Each workshop cell sends its code here and the
# objects (wf, lpad, molecules) persist between cells, exactly like a Jupyter kernel.
shutil.rmtree(SRVDIR, ignore_errors=True); os.makedirs(SRVDIR, exist_ok=True)
server_src = (
"import os, io, time, traceback, contextlib\n"
"NS = {}\n"
"open('" + SRVDIR + "/pid','w').write(str(os.getpid()))\n"
"n = 0\n"
"while True:\n"
"    cmd = '" + SRVDIR + "/cmd_%d.py' % n\n"
"    if not os.path.exists(cmd):\n"
"        time.sleep(0.1); continue\n"
"    code = open(cmd).read(); buf = io.StringIO()\n"
"    try:\n"
"        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):\n"
"            exec(code, NS)\n"
"    except Exception:\n"
"        buf.write(traceback.format_exc())\n"
"    open('" + SRVDIR + "/out_%d.txt' % n, 'w').write(buf.getvalue())\n"
"    open('" + SRVDIR + "/done_%d' % n, 'w').write('1'); n += 1\n"
)
open("/tmp/mispr_server.py","w").write(server_src)
senv = os.environ.copy()
senv["PATH"] = CONDA + "/bin:" + senv["PATH"]
senv["LD_LIBRARY_PATH"] = CONDA + "/lib:" + senv.get("LD_LIBRARY_PATH","")
senv["FW_CONFIG_FILE"] = CAPSULE + "/code/config/FW_config.yaml"
subprocess.Popen([CONDA+"/bin/python","-u","/tmp/mispr_server.py"], env=senv,
                 stdout=open("/tmp/mispr_server.log","w"), stderr=subprocess.STDOUT)

# lines of harmless library noise we don't want students to see
_HIDE = ["will not be saved to the database due to a missing input file"]
def mispr(code):
    """Send code to the persistent Python-3.8 helper and print its output."""
    i = 0
    while os.path.exists(SRVDIR + "/cmd_" + str(i) + ".py"): i += 1
    open(SRVDIR + "/cmd_" + str(i) + ".py", "w").write(code)
    waited = 0.0
    while not os.path.exists(SRVDIR + "/done_" + str(i)):
        time.sleep(0.2); waited += 0.2
        if waited > 900:
            print("[helper timed out - see /tmp/mispr_server.log]"); return
    out = open(SRVDIR + "/out_" + str(i) + ".txt").read()
    lines = [ln for ln in out.splitlines() if not any(h in ln for h in _HIDE)]
    text = "\n".join(lines)
    if text.strip(): print(text)

def show_img(path, width=None, height=None):
    # width only -> the browser preserves the image's real aspect ratio (no stretching)
    from IPython.display import Image, display
    if os.path.exists(path): display(Image(path, width=width))
    else: print("(image not found: " + path + ")")

for _ in range(60):
    if os.path.exists(SRVDIR + "/pid"): break
    time.sleep(0.5)
mispr("import warnings, logging, os\n"
      "warnings.filterwarnings('ignore')\n"
      "logging.getLogger('fireworks').setLevel(logging.WARNING)\n"
      "os.chdir('/root/capsule/results')\n"
      "print('MISPR helper ready')")

print("\n[5/5] Ready in " + str(int(time.time()-t0)) + "s. Work through the three sections below, cell by cell.")


---

# Electrostatic partial charges (on a molecule)

This tutorial focuses on the electrostatic partial workflow in MISPR. This is also a DFT-based workflow but is considered to be simpler in design compared to the previous binding energy workflow because it involves calculations corresponding to a single molecule instead of a set of molecules. 

As the name implies, a partial charge is the net atomic charge. Partial charges are created due to the asymmetric distribution of electrons in chemical bonds.

It is important to calculate partial charges on atoms because they give us an idea of how molecules are expected to bind, e.g. negative dipoles on a molecule will bind strongly with positive dipoles on another. Partial charges are also used in molecular dynamics simulations. 

The ESP workflow in MISPR runs the following steps:
1. Optimizes the geometry of the input structure 
2. Runs a frequency analysis on the optimized structure to make sure the molecule is a real minimum
3. Calculates the partial charges on the atoms 

Steps 1 and 2 are necessary because an optimum geometry will lead to more accurate atomic charges.

## ESP workflow for a water molecule

Instead of running the workflow by providing the structure of the molecule, we will provide a water output file from a previous optimization calculation and will ask the workflow to skip the optimization and frequency jobs described above. 

Remember that we already optimized the structure of water in the previous binding energy workflow, so to avoid running duplicate jobs and consuming computational resources, we use the results from the previous calculation on the same molecule as input to the workflow.

In [ ]:
#@title Build the ESP workflow for water
mispr(r"""
from mispr.gaussian.utilities.fw_utilities import run_fake_gaussian
from mispr.gaussian.workflows.base.esp import get_esp_charges

ref_dir = "/root/capsule/data/esp/water"
wf, _ = get_esp_charges(
    mol_operation_type="get_from_gout_file",
    mol="/root/capsule/data/esp/water/water.out",
    process_mol_func=False,
    mol_name="water",
    skips=["opt", "freq"],
    esp_gaussian_inputs={"route_parameters": {"Density": None}},
    save_to_db=True,
    save_to_file=True,
    working_dir="/root/capsule/results/esp",
)
wf = run_fake_gaussian(
    wf,
    ref_dirs=["/root/capsule/data/esp/water/ESP"],
    input_files=["water_esp.com"],
    tolerance=10,
)
print("ESP workflow for water is ready.")
""")

Similar to before, we will initialize the database and add the workflow to it. We'll also get a summary of the workflow. 

Notice how here we don't see any information about the optimization and frequency fireworks since we skipped those steps.

In [ ]:
#@title Initialize the database and add the workflow
mispr(r"""
from datetime import date
from fireworks import LaunchPad

today = date.today()
d = today.strftime("%Y-%m-%d")

lpad = LaunchPad.auto_load()
lpad.reset(d)
lpad.add_wf(wf)
print(lpad.get_wf_summary_dict(fw_id=1))
""")

We run the workflow using rapidfire of FireWorks.

In [ ]:
#@title Run the workflow
mispr(r"""
from fireworks.core.rocket_launcher import rapidfire
rapidfire(lpad)
print("Workflow finished.")
""")

## Output and visualization

Let us check the calculated charges on the water atoms by reading the summary json file generated by the workflow.

In [ ]:
import json
with open("/root/capsule/results/esp/water/analysis/esp.json") as json_file:
    data = json.load(json_file)
print(data["esp"])

We can use the calculation output files to visualize the charge distribution on the molecule in the form of a heat map. We just need to feed it to a molecule visualization software like Avogadro or VMD. 

The charge distribution on the water molecules looks like this:

In [ ]:
show_img("/root/capsule/data/esp/water/esp_water.png", width=300)

Negatively charged oxygen atom is in red while the positive hydrogens are in blue.

# Running the ESP workflow for multiple molecules

Here we will perform steps similar to the above, but will perform the ESP calculations for multiple molecules using a single code block instead of having to repeat the code multiple times. 

We do that by simply using a for loop and iterating over the molecules we want to calculate the ESP charges for. We will do it for three molecules, but the same thing can be done to a much bigger library.

This example illustrates how we can use MISPR to run calculations in high-throughput mode.

## Getting the workflows

In [ ]:
#@title Run ESP for acetonitrile, cyclohexane, and DMSO
mispr(r"""
esp_gaussian_inputs = {
    "functional": "B3LYP",
    "basis_set": "6-31+G*",
    "route_parameters": {
        "pop": "MK", "iop(6/50=1)": None, "NoSymmetry": None, "Density": None, "test": None,
    },
    "link0_parameters": {"%chk": "esp.chk", "%mem": "45GB", "%NProcShared": "28"},
}
for mol in ["acn", "cyclohexane", "dmso"]:
    wf, _ = get_esp_charges(
        mol_operation_type="get_from_gout_file",
        mol=f"/root/capsule/data/esp/{mol}/{mol}.out",
        process_mol_func=False,
        mol_name=mol,
        skips=["opt", "freq"],
        esp_gaussian_inputs=esp_gaussian_inputs,
        save_to_db=True,
        save_to_file=True,
        working_dir="/root/capsule/results/esp",
    )
    wf = run_fake_gaussian(
        wf,
        ref_dirs=[f"/root/capsule/data/esp/{mol}/ESP"],
        input_files=[f"{mol}_esp.com"],
        tolerance=10,
    )
    lpad.add_wf(wf)
rapidfire(lpad)
print("Workflow finished.")
""")

## Output and visualization

In [ ]:
import json
for mol in ["acn", "cyclohexane", "dmso"]:
    with open(f"/root/capsule/results/esp/{mol}/analysis/esp.json") as json_file:
        data = json.load(json_file)
    print("\n", mol, "\n", data["esp"])

In [ ]:
show_img("/root/capsule/data/esp/esp.png", width=600)

---

# Binding energy between CO₂ and Propionitrile molecule

This tutorial focuses on the binding energy (BE) workflow in MISPR, which represents the recipe that outlines the steps for how to automate binding energy calculations from start to finish. 

There are many preset workflows in MISPR for calculating common properties. To explore all available workflows, you can do that by checking the MISPR source code linked at: 
1. DFT workflows: https://github.com/molmd/mispr/tree/master/mispr/gaussian/workflows/base
2. MD workflows: https://github.com/molmd/mispr/tree/master/mispr/lammps/workflows

For this tutorial, we will focus on the BE workflow, which is a DFT-based calculation. Binding energy is the amount of energy required to bring two particles or molecules together so that they form a bond. It is a critical property in materials science applications because it allows us to understand the interactions between the species forming our systems, thereby providing a way to tailor the system composition according to the desired degree of interactions. 

BE is calculated in MISPR according to the following equation:

$BE = E_{final} - (E_{mol1} + E_{mol2})$

In the above equation, E is the energy on the final complex as well as the energy on each individual molecules *i*. 

The workflows output the BE in the electron volts (eV) unit, but this can be converted to other units like kcal/mol and kJ/mol.

# Binding energy between CO₂ and Propionitrile molecules

# Generating CO₂ structure

We will use the BE workflow in MISPR to calculate the BE between CO₂ and Propionitrile molecules. Let us begin by importing a basic CO₂ structure and taking a look at its structure.

An xyz file is a convenient way to store information about the structure of a molecule. We can examine the contents of an CO₂ file for a CO₂ molecule:

In [ ]:
with open("/root/capsule/data/be/co2.xyz", "r") as f:
    print(f.read())

In [ ]:
with open("/root/capsule/data/be/propionitrile.xyz", "r") as f:
    print(f.read())

We can use the Molecule class from pymatgen to represent the molecule as a python object

In [ ]:
#@title Load the CO₂ structure
mispr(r"""
from pymatgen.core.structure import Molecule
co2_mol = Molecule.from_file("/root/capsule/data/be/co2.xyz")
print(co2_mol)
""")

In [ ]:
show_img("/root/capsule/data/be/Carbon_Dioxide.png", width=400)

In [ ]:
#@title Load the Propionitrile structure
mispr(r"""
from pymatgen.core.structure import Molecule
propionitrile_mol = Molecule.from_file("/root/capsule/data/be/propionitrile.xyz")
print(propionitrile_mol)
""")

In [ ]:
show_img("/root/capsule/data/be/Propionitrile.png", width=400)

MISPR workflows (with the support of FireWorks library) compiles everything needed to: 

* Take and store workflow inputs from the user 
* Create new calculation directories and write input files 
* Parse output directories and store results in a MongoDB database

# Getting workflows

Now, let's get started by importing the BE workflow and then creating a workflow object from CO₂ and Propionitrile structure.

In [ ]:
#@title Build the binding-energy workflow
mispr(r"""
import os
from mispr.gaussian.utilities.fw_utilities import run_fake_gaussian
from mispr.gaussian.workflows.base.binding_energy import get_binding_energies

ref_dir = "/root/capsule/data/be"
working_dir = "/root/capsule/results/be"

wf = get_binding_energies(
    mol_operation_type=["get_from_mol", "get_from_mol"],
    mol=[propionitrile_mol, co2_mol],
    index=[3, 2],
    process_mol_func=False,
    save_to_db=True,
    save_to_file=True,
    mol_name=["propionitrile", "co2"],
    working_dir=working_dir,
    opt_gaussian_inputs={
        "functional": "B3LYP",
        "basis_set": "6-31+G*",
        "route_parameters": {
            "Opt": "(calcfc,tight,cartesian,maxcycles=100)",
            "SCF": "(tight,xqc)",
            "int": "ultrafine",
            "NoSymmetry": None,
            "test": None,
            "EmpiricalDispersion": "GD3",
            "symmetry": "none",
        },
        "link0_parameters": {"%chk": "opt.chk", "%mem": "45GB", "%nprocshared": "28"},
    },
    freq_gaussian_inputs={
        "functional": "B3LYP",
        "basis_set": "6-31+G*",
        "route_parameters": {
            "Freq": None, "iop(7/33=1)": None, "NoSymmetry": None, "test": None,
            "EmpiricalDispersion": "GD3",
        },
        "link0_parameters": {"%chk": "freq.chk", "%mem": "45GB", "%nprocshared": "28"},
    },
)
print(wf)
""")

We can see that this workflow contains seven fireworks, meaning that it is a complex workflow. The exact steps in each firework can vary but for these cases, each firework roughly represents one Gaussian calculation which consists of the following steps:

1. Processing the input molecule 
2. Writing an input file for DFT calculations 
3. Executing commands to run the calculations 
4. Parsing the output files and saving the results in the database 

All these steps are typically done manual and require a lot of effort, but with MISPR, the entire process is simplified to one block of code. 

The workflow will run fireworks for the individual molecules, take the structure of these molecules and create a complex structure, where the molecules bind at a specific site. It will then optimize that structure and calculate its energy. The last firework in this workflow is an analysis step that computes the BE using the equation above.

Before we move forward with running the BE workflow, we will make one modification called run_fake_gaussian. This will change the fireworks in the workflows so they will not actually run Gaussian (which requires greater computer resources and a Gaussian license), but instead will simulate running Gaussian by simply copying over pre-existing Gaussian output files. However, the overall steps in the workflow remain the same as if you are running the calculations on supercomputing resources.

In [ ]:
#@title Replace real Gaussian with the fake runner
mispr(r"""
wf = run_fake_gaussian(
    wf,
    ref_dirs=[
        f"{ref_dir}/propionitrile/Optimization",
        f"{ref_dir}/propionitrile/Frequency",
        f"{ref_dir}/co2/Optimization",
        f"{ref_dir}/co2/Frequency",
        f"{ref_dir}/propionitrile_co2/Optimization",
        f"{ref_dir}/propionitrile_co2/Frequency",
    ],
    input_files=[
        "propionitrile_opt.com", "propionitrile_freq.com",
        "co2_opt.com", "co2_freq.com",
        "propionitrile_co2_opt.com", "propionitrile_co2_freq.com",
    ],
    tolerance=10,
)
print("Fake Gaussian runners set for all 6 reference calculations.")
""")

### Launching workflows from the database

Now we will introduce the LaunchPad from FireWorks which is a python object that let's us interact with MongoDB and manage our calculations. The LaunchPad allows you to submit and query workflows from anywhere you have database access. We need to get ourselves a LaunchPad object and set it up so we can submit our workflows.

In [ ]:
#@title Initialize the LaunchPad
mispr(r"""
from datetime import date
from fireworks import LaunchPad

today = date.today()
d = today.strftime("%Y-%m-%d")

lpad = LaunchPad.auto_load()
lpad.reset(d)
print("LaunchPad initialized.")
""")

Note that lpad.reset() should only be executed one time when you are first initializing your database. If you reset your LaunchPad at a later time, you will erase the record of past calcuations you have run.

Now we will add our generated workflow object into MongoDB so that it can be accessed later.

In [ ]:
#@title Add the workflow to the database
mispr(r"""
lpad.add_wf(wf)
print("Workflow added.")
""")

Now the workflow is added to the database and we can check its status using `.get_wf_summary_dict()`

In [ ]:
#@title Check the workflow status
mispr(r"""
print(lpad.get_wf_summary_dict(fw_id=1))
""")

Now we are ready to run the workflow. Normally we run FireWorks through the command-line interface on the cluster because external codes like Gaussian require more compute resources. However, for today's demonstration, we will run our workflows locally in this notebook to illustrate how it works.

We can run the workflow using the rapidfire mode here.

In [ ]:
#@title Run the workflow
mispr(r"""
from fireworks.core.rocket_launcher import rapidfire
rapidfire(lpad)
print("Workflow finished.")
""")

# Checking the outputs

Let us check the output file (named binding_energy.json) containing a summary of the calculation results. Typically, users will mostly be interested in this final file instead of the intermediate files generated from running the calculations.

In [ ]:
import json
import numpy as np
with open("/root/capsule/results/be/analysis/binding_energy.json") as json_file:
    data = json.load(json_file)
print(data)

We can also open the file in a text editor and check its entries, but let's find out how much is the binding energy between CO₂ and Propionitrile molecules.

In [ ]:
be = np.round(data["be_eV"]["value"] * 23, 2)
print(f"Binding energy between CO2 and Propionitrile molecules is: {be} kcal/mol")

Let us visualize the generated structure.

In [ ]:
show_img("/root/capsule/data/be/co2_propionitrile.png", width=300)

---

# Molecular Dynamics Simulations

So far, you have learned the basics of running DFT workflows in MISPR. We will now move on to running MD simulations in an automated manner by utilizing the workflows in MISPR. 

The goal is to model the behavior of 0.1 M TBA⁺, BF₄⁻ salt and 0.1 M CO₂ in Propionitrile solvent. We will go through both how to set up the input files, as well as how to set up the simulations to minimize and equilibrate the system. Finally we briefly introduce some of the MISPR tools for analyzing molecular simulation trajectories.

# System composition

In [ ]:
show_img("/root/capsule/data/md/composition.png", width=600)

## Structure inputs and initial configuration

The starting point for each simulation is the structure file for each component of the liquid solution. For this tutorial, we will use the structure of TBA⁺, BF₄⁻, CO₂, Propionitrile in the xyz file format. 
From the individual structure files and the number of components of each type, MISPR randomly packs these molecules in the box of the specified length (7 nm here) to prepare an initial configuration to be used in the simulations. 

Let us take a peek at the individual structure files and a visualization of the generated configuration.

In [ ]:
#@title Load the four species
mispr(r"""
from pymatgen.core.structure import Molecule
bf4_mol = Molecule.from_file("/root/capsule/data/md/bf4.xyz"); bf4_mol.set_charge_and_spin(-1, 1)
print(f"bf4 structure\n-------------\n{bf4_mol}")
print("-----------------------------------")
tba_mol = Molecule.from_file("/root/capsule/data/md/tba.xyz"); tba_mol.set_charge_and_spin(1, 1)
print(f"tba structure\n-------------\n{tba_mol}")
print("-----------------------------------")
co2_mol = Molecule.from_file("/root/capsule/data/md/co2.xyz")
print(f"co2 structure\n-------------\n{co2_mol}")
print("-----------------------------------")
propionitrile_mol = Molecule.from_file("/root/capsule/data/md/propionitrile.xyz")
print(f"propionitrile structure\n-------------\n{propionitrile_mol}")
""")

In [ ]:
show_img("/root/capsule/data/md/mixture.png", width=500)

# Data file

To model the atomistic interactions, we need to obtain the force field parameters.

These parameters include nonbonded parameters (atom types and charges) as well as bonded parameters (bonds, angles, dihedrals and atom connectivity).
MISPR supports automatic generation of these parameters through AmberTools (http://ambermd.org) or automatic retrieval from our database. Using these parameters, it generates a data file that contains all the information about the system to run the simulations. To summarize, this information represents:

1. Configuration of the system (i.e., initial atomic positions)
2. Box size
3. Molecular topology (i.e., which atoms are bonded)
4. Force field parameters

In this tutorial, we will directly provide the data file as an input to the workflow instead of asking it to generate it. Generation of the data file is typically done within the workflow and the user only needs to provide the structure of the individual molecules and the composition (i.e, concentration and system dimension) as inputs. However, here we will skip this step so the workflow will not require AmberTools (which if is installed within the capsule, will increase the time required for building the enviornment). 

**Note**: The data file we provide here has been generated before using the same workflow from a previous run.
Now, let's take a look at the first few lines of the data file.

In [ ]:
with open("/root/capsule/data/md/data.mixture") as data_file:
    print(data_file.read(1008))

## Running the MD workflow

Now, we are ready to import the MD workflow and create a workflow object for our electrolyte system.

In [ ]:
#@title Build the MD workflow
mispr(r"""
from mispr.lammps.utilities.utilities import run_fake_lammps
from mispr.lammps.workflows.base import lammps_workflow

template_dir = "/root/capsule/data/md/templates"
ref_dir = "/root/capsule/data/md/runs"

molecules = [tba_mol, propionitrile_mol, bf4_mol, co2_mol]
mol_names = ["tba", "propionitrile", "bf4", "co2"]
num_mols = [20, 2625, 20, 20]
num_atoms_per_mol = [53, 9, 5, 3]
mass = [14.006700, 12.010700, 1.007940, 14.006700, 12.010700, 12.010700,
        1.007940, 10.811000, 18.998403, 15.999400, 12.010700]

wf = lammps_workflow(
    data_file_name="/root/capsule/data/md/data.mixture",
    working_dir="/root/capsule/results/md",
    recipe=[
        ["min", ["template_filename", "min.in"]],
        ["npt", ["template_filename", "npt.in"]],
        ["melt", ["template_filename", "melt.in"]],
        ["quench", ["template_filename", "quench.in"]],
        ["nvt", ["template_filename", "nvt.in"]],
    ],
    analysis_list=["diffusion", "rdf"],
    analysis_settings=[
        {"diff_names": mol_names, "timestep": 1, "com_drift": True, "msd_method": "from_dump",
         "dump_freq": 50000, "diff_dist": True, "avg_interval": True,
         "num_mols": num_mols, "num_atoms_per_mol": num_atoms_per_mol},
        {"rdf_type": "molecular", "partial_relations": [[11, 11, 11], [1, 2, 3]],
         "use_default_atom_ids": True, "mass": mass,
         "num_mols": num_mols, "num_atoms_per_mol": num_atoms_per_mol},
    ],
    template_dir=template_dir,
)
print("MD workflow built (min -> npt -> melt -> quench -> nvt, then analysis).")
""")

The above workflow encapsulates all the steps required to fully automate not only the MD simulations but also the subsequent analysis. 

More specifically, here the workflow will run the following steps:
1. Minimize the system 
2. Equilibrate the system at constant number of particles (N), pressure (P), and temperature (T) - NPT run
3. Melt the system to a higher temperature and then quench it - this is done to ensure no particles are trapped 
4. Run the system at a constant number of particles (N), volume (V), and temperature (T) - NVT run 

After these steps are completed, the workflow will run analysis codes to derive properties of interest from the simulation output and trajectory files. Here, we are requesting to calculate:

1. Diffusion coefficients 
2. Radial distribution functions

Similar to the DFT workflow examples, we will replace the actual MD jobs with fake ones. This will change the fireworks in the workflows so they will not actually run LAMMPS (which requires greater computer resources), but instead will simulate running LAMMPS by simply copying over pre-existing LAMMPS output and trajectory files. However, the overall steps in the workflow remain the same as if you are running the calculations on supercomputing resources.

In [ ]:
#@title Replace real LAMMPS with the fake runner
mispr(r"""
wf = run_fake_lammps(
    wf,
    ref_dirs=[
        f"{ref_dir}/min",
        f"{ref_dir}/npt",
        f"{ref_dir}/melt",
        f"{ref_dir}/quench",
        f"{ref_dir}/nvt",
    ],
)
print("Fake LAMMPS runners set for all 5 stages.")
""")

Now, we are ready to initialize the database and add the workflow to it. We will also get a summary of its status

In [ ]:
#@title Initialize the database and add the workflow
mispr(r"""
from datetime import date
from fireworks import LaunchPad

today = date.today()
d = today.strftime("%Y-%m-%d")
lpad = LaunchPad.auto_load()
lpad.reset(d)
lpad.add_wf(wf)
print(lpad.get_wf_summary_dict(fw_id=1))
""")

Finally, we will run the workflow using rapidfile. This step will take a few moments.

In [ ]:
#@title Run the workflow
mispr(r"""
from fireworks.core.rocket_launcher import rapidfire
rapidfire(lpad)
print("Workflow finished.")
""")

# System visualization

In [ ]:
show_img("/root/capsule/data/md/npt.gif", width=500)

# System properties

# Density

In [ ]:
# density is computed with mdproptools (Python 3.8) and then displayed here
mispr(r"""
from pymatgen.io.lammps.outputs import parse_lammps_log
from mdproptools.utilities.fluctuations import plot_fluctuations
log = parse_lammps_log("/root/capsule/data/md/result/npt/log.lammps")
plot_fluctuations(log[0], "Density", "Density", "density.png",
                  working_dir="/root/capsule/data/md/result")
print("density.png written")
""")
show_img("/root/capsule/data/md/result/density.png", width=600)

**Note**: x-axis unit in the above plot should be seconds and the density on the y-axis is in g/cm3.

# Diffusivity

Diffusion is the movement of molecules from high concentration to low concentration regions. When two concentrations are different in two areas, this is referred to as the concentration gradient, moving concentration from high concentration to low concentration means the substance moves with its concentration gradient.

Diffusivity is a measure of the diffusion or transport of molecules or ions within the system. The higher the diffusivity of a specific molecule, the faster it moves throughout the solution. 

MISPR computes the diffusion of each specie in the system from the atomic coordinates at each time step. It outputs the diffusion coefficients in cm2/s. Let's take a look at the results.

In [ ]:
import pandas as pd
from IPython.display import display
df_diff = pd.read_csv("/root/capsule/data/md/result/analysis/diff/diffusion.csv", index_col=0)
df_diff.columns = ["type", "diffusion (m2/s)", "std", "R2"]
display(df_diff)

# Radial distribution function

Radial distribution function (RDF) represents the probability of finding a particle within a distance of another particle. The higher this probability at short distance, the stronger the interaction is between the two particles. It is calculated using the atomic coordinates at each time step. 
RDF is one of the standard structural properties that can be predicted from MD simulations and it is critical for understanding the atomistic interactions in the system for correlating it to the overall observed behavior of the system.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
df_rdf = pd.read_csv("/root/capsule/data/md/result/analysis/rdf/molecular/0.5/rdf.csv")
plt.plot(df_rdf["r ($\AA$)"], df_rdf["g_11-2"], label="C (CO2) - Propionitrile", color="black", linewidth=1.5)
plt.legend(fontsize=18, frameon=False)
plt.xlabel("distance (Angstroms)", fontsize=18)
plt.ylabel("RDF", fontsize=18)
plt.tick_params(axis="both", which="major", labelsize=16)
plt.show()